[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/091.%20The%20Contract%20Design%20to%20Mitigate%20the%20Bullwhip%20Effect/P91-Tier-10_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# P91-Tier-10: Behavioral Economics Contract Design

## Problem and scope

This notebook builds a stylized behavioral contract-design workflow for a four-tier supply chain. It represents stakeholder behavioral traits, scores a catalog of behavioral nudges, selects nudges using a score-ranked budget heuristic, estimates contract acceptance, and simulates how selected contract features affect orders across three downstream tiers.

The goal is to make the modeling workflow explicit and inspectable. The calculations are illustrative: the behavioral parameters, acceptance function, compliance target, and order-response rules are assumptions for this example, not estimates from field data. The simulation does not establish that any nudge causes a real-world improvement, and its output should not be interpreted as a causal or empirical forecast.

### Inputs and outputs

Inputs are the stakeholder profiles, the specified base contract parameters, the nudge catalog, the scenario budgets, and a seeded synthetic demand series. The workflow reports selected nudges, modeled acceptance scores, simulated compliance and demand-amplification measures, and plots that compare the scenarios.

### Modeling assumptions

- Behavioral traits are represented by fixed numerical profile attributes. They are not inferred from observed stakeholder data.
- Acceptance is a stylized logistic transformation of a constructed utility score. Its numerical values are not calibrated probabilities of real contract acceptance.
- The nudge-selection rule ranks nudges by a behavior-fit and cost-adjusted score, then takes affordable nudges in that order, subject to a four-nudge limit. This is a greedy heuristic; it does not guarantee the globally best combination.
- The simulated chain has one demand series followed by three ordering tiers. Each simulated tier applies the same simple order-response rules.
- Demand is clipped at zero because negative physical demand is not meaningful in this example. No inventory balance, capacity, lead-time dynamics, price response, or strategic interaction is modeled.

## Behavioral representation and contract acceptance

A profile describes how a stakeholder is represented in the model. For example, `present_bias` affects the modeled value of commitment features, while `social_influence` affects the modeled value of peer-comparison features. Traits are descriptive inputs to this stylized rule system, not universal psychological measurements.

For stakeholder $s$, let $u_s$ be the utility score constructed from the monetary contract terms and the behavioral features. The modeled acceptance score is

$$
A_s = \frac{1}{1 + \exp(-u_s/10)}, \qquad
A = \frac{1}{|S|}\sum_{s\in S} A_s.
$$

Here, $S$ is the set of stakeholders, $A_s$ is the modeled acceptance score for stakeholder $s$, and $A$ is the mean across stakeholders. The scale factor $10$ is part of the illustrative logistic mapping. In the code, `_calculate_acceptance` constructs each stakeholder's utility input and `evaluate_contract_option` applies the profile-specific transformations.

## Nudge scoring and budget selection

The designer first calculates average behavioral traits and identifies the largest average trait as the dominant trait. Each nudge begins with its catalog effectiveness score. A behavior-fit multiplier and, when applicable, a complexity multiplier adjust that score. The cost-adjusted ranking score is

$$
q_j = r_j\,\frac{r_j}{c_j + 0.1},
$$

where $r_j$ is the adjusted score for nudge $j$, $c_j$ is its cost, and $0.1$ is the cost denominator offset used by the heuristic. Nudges are sorted by $q_j$ from highest to lowest. In that order, the algorithm selects a nudge if its cost fits the remaining budget, stopping after four selections.

This greedy procedure is deliberately the notebook's selection method: it is straightforward to trace, but it is not an exact optimization algorithm. With $m$ available nudges, scoring takes $O(m)$ work and sorting takes $O(m\log m)$ work. Selection after sorting takes $O(m)$ work.

## Order simulation and interpretation

The simulation starts with demand as the first series. In each of the next three tiers, the base order is demand multiplied by the contract's safety-stock level. Selected behavioral features can adjust that order through loss-framing, commitment smoothing, peer variation, and a points-system compliance adjustment.

For a simulated order $O_{k,t}$ at tier $k$ and period $t$, the example compliance measure compares it with the modeled target $1.1D_{k-1,t}$:

$$
C_{k,t} = \max\!\left(0,\min\!\left(1,
1 - \frac{|O_{k,t}-1.1D_{k-1,t}|}{1.1D_{k-1,t}}\right)\right).
$$

Here, $D_{k-1,t}$ is the incoming series to tier $k$; in the code it is the `demand` value read from the preceding tier's orders. The tier compliance rate is the mean of $C_{k,t}$ over periods. The bullwhip diagnostic is the ratio of coefficients of variation between an ordering tier and the initial demand series. A ratio above one indicates greater relative variability under this diagnostic; it does not by itself identify a cause.

Each simulation has $T$ periods and three ordering tiers, so its principal order calculations take $O(3T)$ work. These results are conditional on the stated simulation rules and seed, not empirical evidence that behavioral contracts reduce the bullwhip effect.

## Workflow

The cells proceed from model components and stakeholder inputs to behavioral analysis, budgeted nudge selection, contract design, simulation, visualization, and interpretation. Run the notebook from top to bottom so that each cell uses the definitions and data created above it.


In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import random

# The fixed seeds make the synthetic demand and simulation randomness reproducible.
np.random.seed(91)
random.seed(91)

print("Behavioral economics contract-design model initialized.")
print("Focus: illustrative behavioral features for bullwhip-effect analysis.")


In [ ]:
@dataclass
class BehavioralProfile:
    """Stylized behavioral attributes for one supply-chain stakeholder."""

    name: str
    risk_tolerance: float
    loss_aversion: float
    present_bias: float
    social_influence: float
    framing_sensitivity: float
    commitment_ability: float
    cognitive_load: float

    def evaluate_contract_option(self, option: Dict[str, Any]) -> float:
        """Transform a constructed contract utility using this profile."""
        base_value = float(option.get("monetary_value", 0.0))

        # Loss aversion applies only to a modeled negative monetary outcome.
        if base_value < 0:
            base_value *= self.loss_aversion

        if "future_value" in option:
            discount_factor = 1.0 - self.present_bias * 0.5
            base_value += float(option["future_value"]) * discount_factor

        if option.get("framing") == "loss_frame":
            base_value *= 1.0 + self.framing_sensitivity * 0.2
        elif option.get("framing") == "gain_frame":
            base_value *= 1.0 - self.framing_sensitivity * 0.1

        if "social_proof" in option:
            base_value *= 1.0 + self.social_influence * float(option["social_proof"]) * 0.3

        # High modeled cognitive load reduces the magnitude of the evaluation.
        if self.cognitive_load > 0.7:
            base_value *= 0.8

        return base_value


@dataclass
class BehavioralNudge:
    """A behavioral intervention in the illustrative nudge catalog."""

    name: str
    nudge_type: str
    description: str
    effectiveness: float
    cost: float
    implementation_complexity: float

    def apply_nudge(
        self,
        base_contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Return a copy of the contract with this nudge's modeled features."""
        modified_contract = base_contract.copy()

        if self.nudge_type == "framing":
            modified_contract = self._apply_framing_nudge(modified_contract, profile)
        elif self.nudge_type == "default":
            modified_contract = self._apply_default_nudge(modified_contract, profile)
        elif self.nudge_type == "social":
            modified_contract = self._apply_social_nudge(modified_contract, profile)
        elif self.nudge_type == "commitment":
            modified_contract = self._apply_commitment_nudge(modified_contract, profile)
        elif self.nudge_type == "gamification":
            modified_contract = self._apply_gamification_nudge(modified_contract, profile)

        return modified_contract

    def _apply_framing_nudge(
        self,
        contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Add modeled loss-framing features when loss aversion is high."""
        if profile.loss_aversion > 1.5:
            contract["framing_effect"] = 0.15
            contract["loss_frame_bonus"] = 0.1
            contract["framing"] = "loss_frame"
        return contract

    def _apply_default_nudge(
        self,
        contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Represent default enrollment for stakeholders with low commitment ability."""
        if profile.commitment_ability < 0.5:
            contract["auto_enrollment"] = 0.8
            contract["opt_out_difficulty"] = 0.3
        return contract

    def _apply_social_nudge(
        self,
        contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Add modeled peer-comparison features when social influence is high."""
        if profile.social_influence > 0.6:
            contract["peer_comparison"] = 0.7
            contract["social_recognition"] = 0.2
            contract["leaderboard_bonus"] = 0.1
        return contract

    def _apply_commitment_nudge(
        self,
        contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Add modeled precommitment features for present-biased stakeholders."""
        if profile.present_bias > 0.5:
            contract["commitment_contract"] = 0.6
            contract["precommitment_bonus"] = 0.15
            contract["penalty_for_break"] = 0.2
        return contract

    def _apply_gamification_nudge(
        self,
        contract: Dict[str, Any],
        profile: BehavioralProfile,
    ) -> Dict[str, Any]:
        """Add the example's points, badge, progress, and achievement features."""
        contract["points_system"] = 0.5
        contract["badge_system"] = 0.3
        contract["progress_bars"] = 0.2
        contract["achievement_unlocks"] = 0.1
        return contract

print("BehavioralProfile and BehavioralNudge classes defined.")


In [ ]:
@dataclass
class BehavioralContractDesigner:
    """Analyze profiles and design contracts using a greedy nudge heuristic."""

    name: str = "BehavioralContractDesigner"
    available_nudges: List[BehavioralNudge] = field(default_factory=list)
    behavioral_insights: Dict[str, Any] = field(default_factory=dict)
    contract_library: List[Dict[str, Any]] = field(default_factory=list)

    def initialize_nudges(self) -> None:
        """Create the fixed nudge catalog used by the example."""
        self.available_nudges = [
            BehavioralNudge("Loss Framing", "framing", "Frame terms as loss prevention", 0.75, 0.1, 0.3),
            BehavioralNudge("Default Options", "default", "Set beneficial options as defaults", 0.68, 0.15, 0.4),
            BehavioralNudge("Social Proof", "social", "Use peer behavior as influence", 0.72, 0.2, 0.5),
            BehavioralNudge("Commitment Devices", "commitment", "Pre-commitment mechanisms", 0.65, 0.25, 0.6),
            BehavioralNudge("Gamification", "gamification", "Game-like elements for engagement", 0.58, 0.3, 0.7),
            BehavioralNudge("Anchoring", "framing", "Set reference points for decisions", 0.63, 0.05, 0.2),
            BehavioralNudge("Mental Accounting", "framing", "Categorize costs and benefits", 0.55, 0.1, 0.3),
            BehavioralNudge("Status Quo Bias", "default", "Leverage preference for the current state", 0.60, 0.12, 0.4),
        ]
        print(f"Initialized {len(self.available_nudges)} behavioral nudges.")

    def analyze_behavioral_patterns(
        self,
        stakeholders: List[BehavioralProfile],
    ) -> Dict[str, Any]:
        """Summarize average traits and their dispersion across stakeholders."""
        if not stakeholders:
            raise ValueError("At least one stakeholder profile is required.")

        trait_values = {
            "loss_aversion": [s.loss_aversion for s in stakeholders],
            "present_bias": [s.present_bias for s in stakeholders],
            "social_influence": [s.social_influence for s in stakeholders],
            "framing_sensitivity": [s.framing_sensitivity for s in stakeholders],
            "commitment_ability": [s.commitment_ability for s in stakeholders],
        }
        averages = {trait: float(np.mean(values)) for trait, values in trait_values.items()}

        patterns: Dict[str, Any] = {
            "loss_aversion_level": averages["loss_aversion"],
            "present_bias_level": averages["present_bias"],
            "social_influence_level": averages["social_influence"],
            "framing_sensitivity_level": averages["framing_sensitivity"],
            "commitment_ability_level": averages["commitment_ability"],
            "dominant_behavioral_trait": max(averages, key=averages.get),
            "behavioral_complexity": float(np.std(list(averages.values()))),
        }
        self.behavioral_insights = patterns
        return patterns

    def select_optimal_nudges(
        self,
        behavioral_patterns: Dict[str, Any],
        budget: float = 1.0,
    ) -> List[BehavioralNudge]:
        """Rank nudges and greedily select affordable items, up to four."""
        if budget < 0:
            raise ValueError("The nudge budget must be non-negative.")

        dominant_trait = behavioral_patterns["dominant_behavioral_trait"]
        complexity = behavioral_patterns["behavioral_complexity"]
        scored_nudges: List[Tuple[BehavioralNudge, float]] = []

        for nudge in self.available_nudges:
            score = nudge.effectiveness

            if dominant_trait == "loss_aversion" and nudge.nudge_type == "framing":
                score *= 1.3
            elif dominant_trait == "present_bias" and nudge.nudge_type == "commitment":
                score *= 1.3
            elif dominant_trait == "social_influence" and nudge.nudge_type == "social":
                score *= 1.3
            elif dominant_trait == "framing_sensitivity" and nudge.nudge_type == "framing":
                score *= 1.2

            if complexity > 0.3:
                score *= 1.2 if nudge.implementation_complexity < 0.4 else 0.8

            # This cost-adjusted score defines the ranking used by the greedy rule.
            score *= score / (nudge.cost + 0.1)
            scored_nudges.append((nudge, score))

        scored_nudges.sort(key=lambda item: item[1], reverse=True)
        selected_nudges: List[BehavioralNudge] = []
        total_cost = 0.0

        for nudge, _ in scored_nudges:
            if total_cost + nudge.cost <= budget:
                selected_nudges.append(nudge)
                total_cost += nudge.cost
                if len(selected_nudges) >= 4:
                    break

        return selected_nudges

    def design_behavioral_contract(
        self,
        base_parameters: Dict[str, Any],
        stakeholders: List[BehavioralProfile],
        budget: float = 1.0,
    ) -> Dict[str, Any]:
        """Apply selected nudges and report modeled acceptance and budget use."""
        if not stakeholders:
            raise ValueError("At least one stakeholder profile is required.")

        behavioral_patterns = self.analyze_behavioral_patterns(stakeholders)
        selected_nudges = self.select_optimal_nudges(behavioral_patterns, budget)
        behavioral_contract: Dict[str, Any] = base_parameters.copy()
        behavioral_contract["applied_nudges"] = []
        behavioral_contract["nudge_effects"] = {}

        for nudge in selected_nudges:
            acceptance_before = self._calculate_acceptance(behavioral_contract, stakeholders)
            modified_contract = nudge.apply_nudge(behavioral_contract, stakeholders[0])
            acceptance_after = self._calculate_acceptance(modified_contract, stakeholders)
            behavioral_contract.update(modified_contract)
            behavioral_contract["applied_nudges"].append(nudge.name)
            behavioral_contract["nudge_effects"][nudge.name] = acceptance_after - acceptance_before

        behavioral_contract["behavioral_effectiveness"] = self._calculate_acceptance(
            behavioral_contract, stakeholders
        )
        behavioral_contract["behavioral_patterns"] = behavioral_patterns
        behavioral_contract["nudge_budget_used"] = sum(n.cost for n in selected_nudges)
        self.contract_library.append(behavioral_contract.copy())
        return behavioral_contract

    def _calculate_acceptance(
        self,
        contract: Dict[str, Any],
        stakeholders: List[BehavioralProfile],
    ) -> float:
        """Compute the model's mean logistic acceptance score."""
        if not stakeholders:
            raise ValueError("At least one stakeholder profile is required.")

        scores = []
        for stakeholder in stakeholders:
            # Contract rates are combined into a stylized utility input; the factor
            # of 2.0 gives modeled behavioral features a visible role in the example.
            monetary_value = (
                float(contract.get("bonus_rate", 0.0))
                + float(contract.get("information_sharing_incentive", 0.0))
                - float(contract.get("penalty_rate", 0.0))
            )
            monetary_value += 2.0 * (
                float(contract.get("framing_effect", 0.0)) * stakeholder.framing_sensitivity
                + float(contract.get("auto_enrollment", 0.0)) * (1.0 - stakeholder.commitment_ability)
                + float(contract.get("commitment_contract", 0.0)) * stakeholder.present_bias
                + float(contract.get("peer_comparison", 0.0)) * stakeholder.social_influence
                + 0.2 * float(contract.get("points_system", 0.0))
            )
            option: Dict[str, Any] = {"monetary_value": monetary_value}

            future_value = (
                float(contract.get("precommitment_bonus", 0.0))
                + 0.1 * float(contract.get("auto_enrollment", 0.0))
                + 0.1 * float(contract.get("points_system", 0.0))
            )
            if future_value:
                option["future_value"] = future_value
            if "framing" in contract:
                option["framing"] = contract["framing"]
            if "peer_comparison" in contract:
                option["social_proof"] = contract["peer_comparison"]

            utility = stakeholder.evaluate_contract_option(option)
            acceptance = 1.0 / (1.0 + np.exp(-utility / 10.0))
            scores.append(float(acceptance))

        return float(np.mean(scores))

print("BehavioralContractDesigner class defined.")


In [ ]:
designer = BehavioralContractDesigner()
designer.initialize_nudges()

print("\nAvailable behavioral nudges:")
for index, nudge in enumerate(designer.available_nudges, start=1):
    print(
        f"{index}. {nudge.name} ({nudge.nudge_type}): "
        f"effectiveness score={nudge.effectiveness:.0%}, cost={nudge.cost:.2f}"
    )
    print(f"   {nudge.description}")


In [ ]:
stakeholders = [
    BehavioralProfile(
        name="Retail Manager",
        risk_tolerance=0.3,
        loss_aversion=2.5,
        present_bias=0.7,
        social_influence=0.6,
        framing_sensitivity=0.8,
        commitment_ability=0.4,
        cognitive_load=0.6,
    ),
    BehavioralProfile(
        name="Wholesaler Director",
        risk_tolerance=0.5,
        loss_aversion=2.0,
        present_bias=0.5,
        social_influence=0.7,
        framing_sensitivity=0.6,
        commitment_ability=0.6,
        cognitive_load=0.5,
    ),
    BehavioralProfile(
        name="Distributor VP",
        risk_tolerance=0.6,
        loss_aversion=1.8,
        present_bias=0.4,
        social_influence=0.5,
        framing_sensitivity=0.5,
        commitment_ability=0.7,
        cognitive_load=0.4,
    ),
    BehavioralProfile(
        name="Manufacturer Executive",
        risk_tolerance=0.7,
        loss_aversion=1.5,
        present_bias=0.3,
        social_influence=0.4,
        framing_sensitivity=0.4,
        commitment_ability=0.8,
        cognitive_load=0.3,
    ),
]

print("Stakeholder behavioral profiles:")
for stakeholder in stakeholders:
    print(f"\n{stakeholder.name}")
    print(f"  Risk tolerance: {stakeholder.risk_tolerance:.0%}")
    print(f"  Loss aversion coefficient: {stakeholder.loss_aversion:.1f}")
    print(f"  Present bias: {stakeholder.present_bias:.0%}")
    print(f"  Social influence: {stakeholder.social_influence:.0%}")
    print(f"  Framing sensitivity: {stakeholder.framing_sensitivity:.0%}")
    print(f"  Commitment ability: {stakeholder.commitment_ability:.0%}")


In [ ]:
behavioral_patterns = designer.analyze_behavioral_patterns(stakeholders)

print("Behavioral pattern summary")
print(f"Average loss aversion: {behavioral_patterns['loss_aversion_level']:.2f}")
print(f"Average present bias: {behavioral_patterns['present_bias_level']:.1%}")
print(f"Average social influence: {behavioral_patterns['social_influence_level']:.1%}")
print(f"Average framing sensitivity: {behavioral_patterns['framing_sensitivity_level']:.1%}")
print(f"Average commitment ability: {behavioral_patterns['commitment_ability_level']:.1%}")
print(f"Behavioral complexity (trait-average standard deviation): {behavioral_patterns['behavioral_complexity']:.3f}")
print(
    "Dominant average trait: "
    f"{behavioral_patterns['dominant_behavioral_trait'].replace('_', ' ')}"
)

print("\nInterpretation under the model's descriptive thresholds:")
if behavioral_patterns["loss_aversion_level"] > 2.0:
    print("Loss aversion is high by this rule; loss-prevention framing may be relevant.")
elif behavioral_patterns["loss_aversion_level"] > 1.5:
    print("Loss aversion is moderate by this rule; both gain and loss framing may be considered.")
else:
    print("Loss aversion is relatively low by this rule; gain framing may be considered.")

if behavioral_patterns["present_bias_level"] > 0.6:
    print("Present bias is high by this rule; commitment features may be relevant.")
elif behavioral_patterns["present_bias_level"] > 0.4:
    print("Present bias is moderate by this rule; immediate and future incentives may both matter.")
else:
    print("Present bias is relatively low by this rule; long-term benefits may be emphasized.")

if behavioral_patterns["social_influence_level"] > 0.6:
    print("Social influence is high by this rule; peer-comparison features may be relevant.")
elif behavioral_patterns["social_influence_level"] > 0.4:
    print("Social influence is moderate by this rule; social proof may be used selectively.")
else:
    print("Social influence is relatively low by this rule; individual incentives may be more relevant.")

if behavioral_patterns["behavioral_complexity"] > 0.3:
    print("The trait averages are dispersed; the heuristic favors simpler nudges.")
else:
    print("The trait averages are less dispersed under this measure.")


In [ ]:
base_contract_params = {
    "revenue_sharing_ratio": 0.5,
    "order_flexibility": 1.0,
    "safety_stock_level": 1.2,
    "information_sharing_incentive": 0.1,
    "lead_time_buffer": 0.1,
    "penalty_rate": 0.15,
    "bonus_rate": 0.1,
}

scenarios = [
    {
        "name": "Conservative Budget",
        "budget": 0.5,
        "description": "Limited budget for behavioral interventions",
    },
    {
        "name": "Moderate Budget",
        "budget": 1.0,
        "description": "Standard budget for behavioral design",
    },
    {
        "name": "Aggressive Budget",
        "budget": 1.5,
        "description": "Higher budget for behavioral interventions",
    },
]

designed_contracts = []
for scenario in scenarios:
    behavioral_contract = designer.design_behavioral_contract(
        base_contract_params,
        stakeholders,
        scenario["budget"],
    )
    designed_contracts.append(
        {
            "scenario": scenario["name"],
            "budget": scenario["budget"],
            "description": scenario["description"],
            "contract": behavioral_contract,
        }
    )

    print(f"\n{scenario['name']}")
    print(f"Description: {scenario['description']}")
    print(f"Budget: {scenario['budget']:.2f}")
    print("Selected nudges:")
    if behavioral_contract["applied_nudges"]:
        for nudge_name in behavioral_contract["applied_nudges"]:
            effect = behavioral_contract["nudge_effects"][nudge_name]
            print(f"  {nudge_name}: modeled acceptance change {effect:+.2%}")
    else:
        print("  No nudges fit the budget.")
    print(
        "Modeled acceptance score: "
        f"{behavioral_contract['behavioral_effectiveness']:.2%}"
    )
    print(
        f"Nudge budget used: {behavioral_contract['nudge_budget_used']:.2f}/"
        f"{scenario['budget']:.2f}"
    )

print("\nContract design scenarios completed.")


In [ ]:
def simulate_behavioral_bullwhip(
    demand_pattern: np.ndarray,
    contract_params: Dict[str, Any],
    behavioral_effects: Dict[str, float],
    n_periods: int = 52,
) -> Tuple[List[np.ndarray], List[float]]:
    """Simulate three ordering tiers and their modeled compliance rates."""
    demand = np.asarray(demand_pattern, dtype=float)[:n_periods]
    if demand.size == 0:
        raise ValueError("The demand pattern must contain at least one period.")
    if np.any(demand < 0):
        raise ValueError("Demand must be non-negative before simulation.")

    orders = [demand.copy()]
    compliance_rates = []

    for _ in range(3):
        previous_orders = orders[-1]
        current_orders = []
        tier_compliance = []

        for period, incoming_demand in enumerate(previous_orders):
            base_order = incoming_demand * contract_params.get("safety_stock_level", 1.2)
            behavioral_adjustment = 1.0

            if "loss_frame_bonus" in behavioral_effects:
                behavioral_adjustment *= 1.0 - 0.1 * behavioral_effects["loss_frame_bonus"]

            if "commitment_contract" in behavioral_effects and period > 0:
                smoothing = 0.3 * behavioral_effects["commitment_contract"]
                base_order = (
                    smoothing * base_order
                    + (1.0 - smoothing) * current_orders[-1]
                )

            if "peer_comparison" in behavioral_effects and period > 0:
                peer_influence = 0.1 * behavioral_effects["peer_comparison"]
                base_order *= 1.0 + peer_influence * np.random.normal(0.0, 0.1)

            if "points_system" in behavioral_effects:
                compliance_bonus = 0.05 * behavioral_effects["points_system"]
                behavioral_adjustment *= 1.0 + compliance_bonus

            final_order = max(0.0, base_order * behavioral_adjustment)
            current_orders.append(final_order)

            optimal_order = incoming_demand * 1.1
            if optimal_order == 0.0:
                compliance = 1.0 if final_order == 0.0 else 0.0
            else:
                compliance = 1.0 - abs(final_order - optimal_order) / optimal_order
                compliance = max(0.0, min(1.0, compliance))
            tier_compliance.append(compliance)

        orders.append(np.asarray(current_orders, dtype=float))
        compliance_rates.append(float(np.mean(tier_compliance)))

    return orders, compliance_rates


# The seed set in the first code cell makes this synthetic demand reproducible.
test_demand = np.maximum(np.random.normal(100.0, 25.0, 52), 0.0)
comparison_results = []

traditional_orders, traditional_compliance = simulate_behavioral_bullwhip(
    test_demand,
    base_contract_params,
    {},
)

for contract_result in designed_contracts:
    contract = contract_result["contract"]
    behavioral_effects = {
        key: float(contract[key])
        for key in (
            "loss_frame_bonus",
            "commitment_contract",
            "peer_comparison",
            "points_system",
        )
        if key in contract
    }
    behavioral_orders, behavioral_compliance = simulate_behavioral_bullwhip(
        test_demand,
        base_contract_params,
        behavioral_effects,
    )
    comparison_results.append(
        {
            "type": contract_result["scenario"],
            "orders": behavioral_orders,
            "compliance": behavioral_compliance,
            "effectiveness": contract["behavioral_effectiveness"],
        }
    )

print("Simulation comparison")
print(f"Traditional mean tier compliance: {np.mean(traditional_compliance):.2%}")
for result in comparison_results:
    modeled_compliance = float(np.mean(result["compliance"]))
    baseline_compliance = float(np.mean(traditional_compliance))
    relative_change = (
        (modeled_compliance / baseline_compliance - 1.0) * 100.0
        if baseline_compliance != 0.0
        else float("nan")
    )
    print(f"{result['type']} mean tier compliance: {modeled_compliance:.2%}")
    print(f"  Relative change from traditional simulation: {relative_change:+.1f}%")

print(
    "Interpret these values as outputs of the stated simulation rules, "
    "not as observed contract effects."
)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Behavioral Contract Design: Model Outputs", fontsize=16, fontweight="bold")

# Compare the six profile traits; loss aversion is scaled for this shared 0-to-1 display.
ax1 = axes[0, 0]
traits = [
    "Risk\nTolerance",
    "Loss\nAversion",
    "Present\nBias",
    "Social\nInfluence",
    "Framing\nSensitivity",
    "Commitment\nAbility",
]
trait_values = [
    [profile.risk_tolerance for profile in stakeholders],
    [profile.loss_aversion / 3.0 for profile in stakeholders],
    [profile.present_bias for profile in stakeholders],
    [profile.social_influence for profile in stakeholders],
    [profile.framing_sensitivity for profile in stakeholders],
    [profile.commitment_ability for profile in stakeholders],
]
x = np.arange(len(traits))
width = 0.16
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4"]
for index, stakeholder in enumerate(stakeholders):
    ax1.bar(
        x + index * width,
        [trait_values[trait_index][index] for trait_index in range(len(traits))],
        width,
        label=stakeholder.name,
        color=colors[index],
        alpha=0.85,
    )
ax1.set_xlabel("Behavioral trait")
ax1.set_ylabel("Displayed level (0–1; loss aversion divided by 3)")
ax1.set_title("Stakeholder profile inputs")
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(traits)
ax1.legend(fontsize=8)
ax1.grid(True, axis="y", alpha=0.3)

# Compare modeled acceptance scores with the base contract's score.
ax2 = axes[0, 1]
scenario_names = [item["scenario"] for item in designed_contracts]
effectiveness_scores = [
    item["contract"]["behavioral_effectiveness"] for item in designed_contracts
]
budgets_used = [item["contract"]["nudge_budget_used"] for item in designed_contracts]
baseline_acceptance = designer._calculate_acceptance(base_contract_params, stakeholders)
bars = ax2.bar(range(len(scenario_names)), effectiveness_scores, color=colors[:len(scenario_names)])
ax2.axhline(
    y=baseline_acceptance,
    color="gray",
    linestyle="--",
    label="Base contract modeled acceptance",
)
ax2.set_xlabel("Budget scenario")
ax2.set_ylabel("Modeled acceptance score")
ax2.set_title("Modeled contract acceptance")
ax2.set_ylim(0.0, 1.0)
ax2.set_xticks(range(len(scenario_names)))
ax2.set_xticklabels([name.replace(" ", "\n") for name in scenario_names])
ax2.legend(fontsize=8)
ax2.grid(True, axis="y", alpha=0.3)
for bar, score, budget_used in zip(bars, effectiveness_scores, budgets_used):
    ax2.text(
        bar.get_x() + bar.get_width() / 2.0,
        min(bar.get_height() + 0.015, 0.98),
        f"{score:.1%}\n(budget {budget_used:.2f})",
        ha="center",
        va="bottom",
        fontsize=8,
    )

# Show how often each nudge was selected across the budget scenarios.
ax3 = axes[0, 2]
nudge_counts = defaultdict(int)
for item in designed_contracts:
    for nudge_name in item["contract"]["applied_nudges"]:
        nudge_counts[nudge_name] += 1
nudge_names = list(nudge_counts)
nudge_frequencies = [nudge_counts[name] for name in nudge_names]
if nudge_names:
    bars = ax3.barh(nudge_names, nudge_frequencies, color="#96CEB4")
    for bar, frequency in zip(bars, nudge_frequencies):
        ax3.text(
            bar.get_width() + 0.03,
            bar.get_y() + bar.get_height() / 2.0,
            str(frequency),
            va="center",
        )
else:
    ax3.text(0.5, 0.5, "No nudges selected", ha="center", va="center")
ax3.set_xlabel("Number of scenarios")
ax3.set_ylabel("Nudge")
ax3.set_title("Nudge selection frequency")
ax3.grid(True, axis="x", alpha=0.3)

# Coefficient-of-variation ratios compare tier variability with initial demand variability.
def calculate_amplification(order_series: List[np.ndarray]) -> List[float]:
    """Return tier coefficients of variation relative to initial demand."""
    base_mean = float(np.mean(order_series[0]))
    if base_mean == 0.0:
        return [float("nan")] * (len(order_series) - 1)
    base_cv = float(np.std(order_series[0]) / base_mean)
    if base_cv == 0.0:
        return [float("nan")] * (len(order_series) - 1)

    amplification = []
    for series in order_series[1:]:
        mean_value = float(np.mean(series))
        if mean_value == 0.0:
            amplification.append(float("nan"))
        else:
            current_cv = float(np.std(series) / mean_value)
            amplification.append(current_cv / base_cv)
    return amplification

ax4 = axes[1, 0]
traditional_amplification = calculate_amplification(traditional_orders)
behavioral_amplifications = [
    calculate_amplification(result["orders"]) for result in comparison_results
]
tiers = range(2, 5)
ax4.plot(
    tiers,
    traditional_amplification,
    "o-",
    linewidth=2,
    color="gray",
    label="Traditional contract",
)
for index, result in enumerate(comparison_results):
    ax4.plot(
        tiers,
        behavioral_amplifications[index],
        "o-",
        linewidth=2,
        color=colors[index],
        label=result["type"],
    )
ax4.axhline(1.0, color="black", linestyle="--", alpha=0.5, label="No relative amplification")
ax4.set_xlabel("Ordering tier")
ax4.set_ylabel("Coefficient-of-variation ratio")
ax4.set_title("Demand variability diagnostic")
ax4.set_xticks(list(tiers))
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# Compliance is reported only for the three simulated ordering tiers.
ax5 = axes[1, 1]
tier_labels = ["Tier 2", "Tier 3", "Tier 4"]
x = np.arange(len(tier_labels))
bar_width = 0.2
ax5.bar(
    x - 1.5 * bar_width,
    traditional_compliance,
    bar_width,
    label="Traditional contract",
    color="gray",
    alpha=0.8,
)
for index, result in enumerate(comparison_results):
    ax5.bar(
        x + (index - 0.5) * bar_width,
        result["compliance"],
        bar_width,
        label=result["type"],
        color=colors[index],
        alpha=0.85,
    )
ax5.set_xlabel("Simulated ordering tier")
ax5.set_ylabel("Modeled compliance rate")
ax5.set_title("Compliance by tier")
ax5.set_xticks(x)
ax5.set_xticklabels(tier_labels)
ax5.set_ylim(0.0, 1.0)
ax5.legend(fontsize=8)
ax5.grid(True, axis="y", alpha=0.3)

# This plot displays modeled acceptance against budget use, not a financial ROI.
ax6 = axes[1, 2]
for index, item in enumerate(designed_contracts):
    contract = item["contract"]
    ax6.scatter(
        contract["nudge_budget_used"],
        contract["behavioral_effectiveness"],
        s=90,
        color=colors[index],
        label=item["scenario"],
        edgecolors="black",
    )
    ax6.annotate(
        item["scenario"],
        (contract["nudge_budget_used"], contract["behavioral_effectiveness"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
ax6.set_xlabel("Nudge budget used (model units)")
ax6.set_ylabel("Modeled acceptance score")
ax6.set_title("Budget use and modeled acceptance")
ax6.set_ylim(0.0, 1.0)
ax6.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(
    "The charts display outputs of the illustrative profile, selection, "
    "acceptance, and simulation rules."
)


In [ ]:
print("Implementation considerations")
print("1. Assess stakeholder behavior using transparent, consent-based data collection.")
print("2. Treat profile attributes and thresholds as assumptions unless validated with evidence.")
print("3. Explain contract terms clearly and preserve meaningful opt-out choices.")
print("4. Monitor realized ordering, service, cost, and compliance outcomes after implementation.")
print("5. Review whether peer comparisons, defaults, and commitment features are appropriate and ethical.")
print(
    "6. Use observed operational data to evaluate whether the modeled relationships "
    "are relevant in the setting of interest."
)

best_contract = max(
    designed_contracts,
    key=lambda item: item["contract"]["behavioral_effectiveness"],
)
print("\nBest modeled acceptance among the listed scenarios:")
print(f"Scenario: {best_contract['scenario']}")
print(
    "Modeled acceptance score: "
    f"{best_contract['contract']['behavioral_effectiveness']:.2%}"
)
print(
    "This ranking is conditional on the illustrative scoring rules and "
    "does not establish real-world effectiveness."
)


In [ ]:
average_effectiveness = float(
    np.mean([item["contract"]["behavioral_effectiveness"] for item in designed_contracts])
)
average_budget_used = float(
    np.mean([item["contract"]["nudge_budget_used"] for item in designed_contracts])
)

baseline_compliance = float(np.mean(traditional_compliance))
scenario_compliance = [
    float(np.mean(result["compliance"])) for result in comparison_results
]
if baseline_compliance != 0.0 and scenario_compliance:
    average_relative_compliance_change = float(
        np.mean(
            [
                (value / baseline_compliance - 1.0) * 100.0
                for value in scenario_compliance
            ]
        )
    )
else:
    average_relative_compliance_change = float("nan")

traditional_amplification = calculate_amplification(traditional_orders)
behavioral_amplifications = [
    calculate_amplification(result["orders"]) for result in comparison_results
]
traditional_final_amplification = traditional_amplification[-1]
behavioral_final_amplifications = [values[-1] for values in behavioral_amplifications]
finite_behavioral_amplifications = [
    value for value in behavioral_final_amplifications if np.isfinite(value)
]
if (
    np.isfinite(traditional_final_amplification)
    and traditional_final_amplification != 0.0
    and finite_behavioral_amplifications
):
    mean_relative_amplification_change = (
        np.mean(finite_behavioral_amplifications) / traditional_final_amplification - 1.0
    ) * 100.0
else:
    mean_relative_amplification_change = float("nan")

print("Summary of the illustrative model")
print(f"Mean modeled acceptance across budget scenarios: {average_effectiveness:.2%}")
print(f"Mean nudge budget used: {average_budget_used:.2f} model units")
print(
    "Mean relative compliance change versus the traditional simulation: "
    f"{average_relative_compliance_change:+.2f}%"
)
print(
    "Mean relative change in final-tier variability ratio versus the traditional "
    f"simulation: {mean_relative_amplification_change:+.2f}%"
)

print("\nInterpretation")
print(
    "Acceptance, compliance, and variability are distinct model outputs. "
    "A change in one does not imply a change in the others."
)
print(
    "The greedy selection scores and simulation rules are assumptions; "
    "the numerical outputs are not validated causal effects, empirical forecasts, "
    "or measured implementation results."
)
print(
    "Assess any real contract using relevant operational data, stakeholder "
    "feedback, and appropriate ethical safeguards."
)
